<a href="https://colab.research.google.com/github/DimiGretsistas/lab-extractive-question-answering/blob/main/lab_extractive_question_answering_finished.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LAB | Extractive Question Answering

This notebook demonstrates how Pinecone helps you build an extractive question-answering application. To build an extractive question-answering system, we need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A reader model to extract answers

We will use the SQuAD dataset, which consists of **questions** and **context** paragraphs containing question **answers**. We generate embeddings for the context passages using the retriever, index them in the vector database, and query with semantic search to retrieve the top k most relevant contexts containing potential answers to our question. We then use the reader model to extract the answers from the returned contexts.

Let's get started by installing the packages needed for notebook to run:

In [21]:
!pip uninstall torchcodec -y -q

In [22]:
!pip install torchvision==0.26.0+cu130 \
    --index-url https://download.pytorch.org/whl/cu130 \
    --force-reinstall -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 67.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 72.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 77.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 79.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 70.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 179.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 55.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 16.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 16.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 43.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.9/145.9 MB 64.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [23]:
import torch, torchvision
print(f"PyTorch:     {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")
print(f"CUDA:        {torch.cuda.is_available()}")
# Both should end in +cu130 ✅

PyTorch:     2.11.0+cu130
torchvision: 0.26.0+cu130
CUDA:        True


# Install Dependencies

In [24]:
# uninstall torchcodec before any imports
!pip uninstall torchcodec -y -q

In [25]:
# now safe to import
!pip install -qU datasets pinecone-client sentence-transformers transformers
!pip install -qU langchain-pinecone pinecone-notebooks

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.7/588.7 kB 44.4 MB/s eta 0:00:00


In [26]:
import os
from google.colab import userdata
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')

# Load Dataset

Now let's load the SQUAD dataset from the HuggingFace Model Hub. We load the dataset into a pandas dataframe and filter the title, question, and context columns, and we drop any duplicate context passages.

In [ ]:
from datasets import load_dataset

# load the squad dataset into a pandas dataframe
df = load_dataset("squad", split="train").to_pandas()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# select only title and context column
df = df[["title", "context"]]
# drop rows containing duplicate context passages
df = df.drop_duplicates(subset=["context"])
# reset the index
df = df.reset_index(drop=True)
df



,title,context
0,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha..."
1,University_of_Notre_Dame,"As at most other universities, Notre Dame's st..."
2,University_of_Notre_Dame,The university is the major seat of the Congre...
3,University_of_Notre_Dame,The College of Engineering was established in ...
4,University_of_Notre_Dame,All of Notre Dame's undergraduate students are...
...,...,...
18886,Kathmandu,"Institute of Medicine, the central college of ..."
18887,Kathmandu,Football and Cricket are the most popular spor...
18888,Kathmandu,The total length of roads in Nepal is recorded...
18889,Kathmandu,The main international airport serving Kathman...


# Initialize Pinecone Index

The Pinecone index stores vector representations of our context passages which we can retrieve using another vector (query vector). We first need to initialize our connection to Pinecone to create our vector index. For this, we need a free [API key]("https://app.pinecone.io/"), and then we initialize the connection like so:

In [ ]:
from pinecone import Pinecone, ServerlessSpec

spec = ServerlessSpec(
    cloud="aws", region="us-east-1"
)

# # connect to pinecone environment
pc = Pinecone(
     api_key = PINECONE_API_KEY,
 )



Now we create a new index called "question-answering" — we can name the index anything we want. We specify the metric type as "cosine" and dimension as 384 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 384-dimension vectors.

In [ ]:
print(pc.list_indexes().names())

[]


In [46]:
from pinecone import Pinecone, ServerlessSpec
import time

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "question-answering"

# check if the index exists
if index_name not in pc.list_indexes().names():

    # create the index if it does not exist
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)

print(index)

# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all context passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will generate embeddings in a way that the questions and context passages containing answers to our questions are nearby in the vector space. We can use cosine similarity to calculate the similarity between the query and context embeddings to find the context passages that contain potential answers to our question.

We will use a SentenceTransformer model named ``multi-qa-MiniLM-L6-cos-v1`` designed for semantic search and trained on 215M (question, answer) pairs from diverse sources as our retriever.

In [47]:
import torch
from sentence_transformers import SentenceTransformer

# set device to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# load the retriever model from huggingface model hub
retriever = SentenceTransformer(
    'multi-qa-MiniLM-L6-cos-v1',
    device=device
)
retriever

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, context passage, etc.

In [48]:
from tqdm.auto import tqdm

batch_size = 64

for i in tqdm(range(0, len(df), batch_size)):
    # find end of batch
    i_end = min(i + batch_size, len(df))

    # extract batch
    batch = df.iloc[i:i_end]

    # generate embeddings for batch
    emb = retriever.encode(batch["context"].tolist()).tolist()

    # get metadata
    meta = batch.to_dict(orient="records")

    # create unique IDs
    ids = [str(x) for x in range(i, i_end)]

    # add all to upsert list
    to_upsert = list(zip(ids, emb, meta))

    # upsert/insert these records to pinecone
    _ = index.upsert(vectors=to_upsert)

index.describe_index_stats()

  0%|          | 0/296 [00:00<?, ?it/s]

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 18891}},
 'total_vector_count': 18891,
 'vector_type': 'dense'}

# Initialize Reader

We use the `deepset/electra-base-squad2` model from the HuggingFace model hub as our reader model. We load this model into a "question-answering" pipeline from HuggingFace transformers and feed it our questions and context passages individually. The model gives a prediction for each context we pass through the pipeline.

In [49]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

model_name = 'deepset/electra-base-squad2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(model_name).to(device)

def reader(question, context):
    inputs = tokenizer(question, context, return_tensors="pt",
                      truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = qa_model(**inputs)
    start = outputs.start_logits.argmax()
    end = outputs.end_logits.argmax() + 1
    answer = tokenizer.convert_tokens_to_string(
        tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][start:end])
    )
    score = float(outputs.start_logits.max() + outputs.end_logits.max())
    return {"answer": answer, "score": score}

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Now all the components we need are ready. Let's write some helper functions to execute our queries. The `get_context` function retrieves the context embeddings containing answers to our question from the Pinecone index, and the `extract_answer` function extracts the answers from these context passages.

In [50]:
from pprint import pprint

def get_context(question, top_k):
    xq = retriever.encode([question]).tolist()
    xc = index.query(vector=xq, top_k=top_k, include_metadata=True)
    return [match["metadata"]["context"] for match in xc["matches"]]


In [51]:
from pprint import pprint

def extract_answer(question, context):
    results = []
    for c in context:
        answer = reader(question=question, context=c)
        answer["context"] = c
        results.append(answer)
    sorted_result = sorted(results, key=lambda x: x['score'], reverse=True)
    pprint(sorted_result)
    return sorted_result

In [52]:
question = "How much oil is Egypt producing in a day?"
context = get_context(question, top_k = 1)
context

['Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of natural gas (in 2013), which makes Egypt as the largest oil producer not member of the Organization of the Petroleum Exporting Countries (OPEC) and the second-largest dry natural gas producer in Africa. In 2013, Egypt was the largest consumer of oil and natural gas in Africa, as more than 20% of total oil consumption and more than 40% of total dry natural gas consumption in Africa. Also, Egypt possesses the largest oil refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is currently planning to build its first nuclear power plant in El Dabaa city, northern Egypt.']

In [53]:
question = "How much oil is Egypt producing in a day?"
context = get_context(question, top_k = 1)
context

['Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of natural gas (in 2013), which makes Egypt as the largest oil producer not member of the Organization of the Petroleum Exporting Countries (OPEC) and the second-largest dry natural gas producer in Africa. In 2013, Egypt was the largest consumer of oil and natural gas in Africa, as more than 20% of total oil consumption and more than 40% of total dry natural gas consumption in Africa. Also, Egypt possesses the largest oil refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is currently planning to build its first nuclear power plant in El Dabaa city, northern Egypt.']

As we can see, the retiever is working fine and gets us the context passage that contains the answer to our question. Now let's use the reader to extract the exact answer from the context passage.

In [54]:
extract_answer(question, context)

[{'answer': '691, 000 bbl / d',
  'context': 'Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of '
             'natural gas (in 2013), which makes Egypt as the largest oil '
             'producer not member of the Organization of the Petroleum '
             'Exporting Countries (OPEC) and the second-largest dry natural '
             'gas producer in Africa. In 2013, Egypt was the largest consumer '
             'of oil and natural gas in Africa, as more than 20% of total oil '
             'consumption and more than 40% of total dry natural gas '
             'consumption in Africa. Also, Egypt possesses the largest oil '
             'refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is '
             'currently planning to build its first nuclear power plant in El '
             'Dabaa city, northern Egypt.',
  'score': 21.08501434326172}]


[{'answer': '691, 000 bbl / d',
  'score': 21.08501434326172,
  'context': 'Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of natural gas (in 2013), which makes Egypt as the largest oil producer not member of the Organization of the Petroleum Exporting Countries (OPEC) and the second-largest dry natural gas producer in Africa. In 2013, Egypt was the largest consumer of oil and natural gas in Africa, as more than 20% of total oil consumption and more than 40% of total dry natural gas consumption in Africa. Also, Egypt possesses the largest oil refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is currently planning to build its first nuclear power plant in El Dabaa city, northern Egypt.'}]

The reader model predicted with 99% accuracy the correct answer *691,000 bbl/d* as seen from the context passage. Let's run few more queries.

In [55]:
question = "What are the first names of the men that invented youtube?"
context = get_context(question, top_k=1)
extract_answer(question, context)

[{'answer': 'hurley and chen',
  'context': 'According to a story that has often been repeated in the media, '
             'Hurley and Chen developed the idea for YouTube during the early '
             'months of 2005, after they had experienced difficulty sharing '
             "videos that had been shot at a dinner party at Chen's apartment "
             'in San Francisco. Karim did not attend the party and denied that '
             'it had occurred, but Chen commented that the idea that YouTube '
             'was founded after a dinner party "was probably very strengthened '
             'by marketing ideas around creating a story that was very '
             'digestible".',
  'score': 19.325923919677734}]


[{'answer': 'hurley and chen',
  'score': 19.325923919677734,
  'context': 'According to a story that has often been repeated in the media, Hurley and Chen developed the idea for YouTube during the early months of 2005, after they had experienced difficulty sharing videos that had been shot at a dinner party at Chen\'s apartment in San Francisco. Karim did not attend the party and denied that it had occurred, but Chen commented that the idea that YouTube was founded after a dinner party "was probably very strengthened by marketing ideas around creating a story that was very digestible".'}]

In [56]:
question = "What is Albert Eistein famous for?"
context = get_context(question, top_k=1)
extract_answer(question, context)

[{'answer': 'his theories of special relativity and general relativity',
  'context': 'Albert Einstein is known for his theories of special relativity '
             'and general relativity. He also made important contributions to '
             'statistical mechanics, especially his mathematical treatment of '
             'Brownian motion, his resolution of the paradox of specific '
             'heats, and his connection of fluctuations and dissipation. '
             'Despite his reservations about its interpretation, Einstein also '
             'made contributions to quantum mechanics and, indirectly, quantum '
             'field theory, primarily through his theoretical studies of the '
             'photon.',
  'score': 12.574865341186523}]


[{'answer': 'his theories of special relativity and general relativity',
  'score': 12.574865341186523,
  'context': 'Albert Einstein is known for his theories of special relativity and general relativity. He also made important contributions to statistical mechanics, especially his mathematical treatment of Brownian motion, his resolution of the paradox of specific heats, and his connection of fluctuations and dissipation. Despite his reservations about its interpretation, Einstein also made contributions to quantum mechanics and, indirectly, quantum field theory, primarily through his theoretical studies of the photon.'}]

Let's run another question. This time for top 3 context passages from the retriever.

In [57]:
question = "Who was the first person to step foot on the moon?"
context = get_context(question, top_k=3)
extract_answer(question, context)

[{'answer': 'armstrong',
  'context': 'The trip to the Moon took just over three days. After achieving '
             'orbit, Armstrong and Aldrin transferred into the Lunar Module, '
             'named Eagle, and after a landing gear inspection by Collins '
             'remaining in the Command/Service Module Columbia, began their '
             'descent. After overcoming several computer overload alarms '
             'caused by an antenna switch left in the wrong position, and a '
             'slight downrange error, Armstrong took over manual flight '
             'control at about 180 meters (590 ft), and guided the Lunar '
             'Module to a safe landing spot at 20:18:04 UTC, July 20, 1969 '
             '(3:17:04 pm CDT). The first humans on the Moon would wait '
             'another six hours before they ventured out of their craft. At '
             '02:56 UTC, July 21 (9:56 pm CDT July 20), Armstrong became the '
             'first human to set foot on the Moon.',

[{'answer': 'armstrong',
  'score': 11.328462600708008,
  'context': 'The trip to the Moon took just over three days. After achieving orbit, Armstrong and Aldrin transferred into the Lunar Module, named Eagle, and after a landing gear inspection by Collins remaining in the Command/Service Module Columbia, began their descent. After overcoming several computer overload alarms caused by an antenna switch left in the wrong position, and a slight downrange error, Armstrong took over manual flight control at about 180 meters (590 ft), and guided the Lunar Module to a safe landing spot at 20:18:04 UTC, July 20, 1969 (3:17:04 pm CDT). The first humans on the Moon would wait another six hours before they ventured out of their craft. At 02:56 UTC, July 21 (9:56 pm CDT July 20), Armstrong became the first human to set foot on the Moon.'},
 {'answer': 'frank borman',
  'score': 4.50399923324585,
  'context': "On December 21, 1968, Frank Borman, James Lovell, and William Anders became the first hu

The result looks pretty good.

In [41]:
pc.delete_index(index_name)

### Add a few more questions. What did you observe?

In [58]:
question = "Who founded Microsoft?"
context = get_context(question, top_k=3)
extract_answer(question, context)

[{'answer': '[SEP]',
  'context': 'The first web browser was invented in 1990 by Sir Tim '
             'Berners-Lee. Berners-Lee is the director of the World Wide Web '
             "Consortium (W3C), which oversees the Web's continued "
             'development, and is also the founder of the World Wide Web '
             'Foundation. His browser was called WorldWideWeb and later '
             'renamed Nexus.',
  'score': 14.340497970581055},
 {'answer': '[SEP]',
  'context': 'Several notable video game developers criticized Microsoft for '
             'making its Windows Store a closed platform subject to its own '
             'regulations, as it conflicted with their view of the PC as an '
             'open platform. Markus "Notch" Persson (creator of the indie game '
             'Minecraft), Gabe Newell (co-founder of Valve Corporation and '
             'developer of software distribution platform Steam), and Rob '
             'Pardo from Activision Blizzard voiced concern

[{'answer': '[SEP]',
  'score': 14.340497970581055,
  'context': "The first web browser was invented in 1990 by Sir Tim Berners-Lee. Berners-Lee is the director of the World Wide Web Consortium (W3C), which oversees the Web's continued development, and is also the founder of the World Wide Web Foundation. His browser was called WorldWideWeb and later renamed Nexus."},
 {'answer': '[SEP]',
  'score': 14.128767013549805,
  'context': 'Several notable video game developers criticized Microsoft for making its Windows Store a closed platform subject to its own regulations, as it conflicted with their view of the PC as an open platform. Markus "Notch" Persson (creator of the indie game Minecraft), Gabe Newell (co-founder of Valve Corporation and developer of software distribution platform Steam), and Rob Pardo from Activision Blizzard voiced concern about the closed nature of the Windows Store. However, Tom Warren of The Verge stated that Microsoft\'s addition of the Store was simply respond

- For this question, the system did not work well. The retriever returned context passages that did not contain the correct answer.
- Because of that, the reader model could not extract a meaningful answer and returned invalid outputs such as [SEP].This shows that the reader depends completely on the quality of the retrieved context.
- If the correct context is not retrieved, the answer will be wrong even if the reader model works correctly.

In [59]:
question = "When and why did World War II end?"
context = get_context(question, top_k=3)
extract_answer(question, context)

[{'answer': '[SEP]',
  'context': 'The First World War began in 1914 and lasted to the final '
             'Armistice in 1918. The Allied Powers, led by the British Empire, '
             'France, Russia until March 1918, Japan and the United States '
             'after 1917, defeated the Central Powers, led by the German '
             'Empire, Austro-Hungarian Empire and the Ottoman Empire. The war '
             'caused the disintegration of four empires—the Austro-Hungarian, '
             'German, Ottoman, and Russian ones—as well as radical change in '
             'the European and West Asian maps. The Allied powers before 1917 '
             'are referred to as the Triple Entente, and the Central Powers '
             'are referred to as the Triple Alliance.',
  'score': 13.933052062988281},
 {'answer': '[CLS]',
  'context': 'The outbreak of World War I in 1914 was precipitated by the rise '
             'of nationalism in Southeastern Europe as the Great Powers took '
      

[{'answer': '[SEP]',
  'score': 13.933052062988281,
  'context': 'The First World War began in 1914 and lasted to the final Armistice in 1918. The Allied Powers, led by the British Empire, France, Russia until March 1918, Japan and the United States after 1917, defeated the Central Powers, led by the German Empire, Austro-Hungarian Empire and the Ottoman Empire. The war caused the disintegration of four empires—the Austro-Hungarian, German, Ottoman, and Russian ones—as well as radical change in the European and West Asian maps. The Allied powers before 1917 are referred to as the Triple Entente, and the Central Powers are referred to as the Triple Alliance.'},
 {'answer': '[CLS]',
  'score': 12.928281784057617,
  'context': 'The outbreak of World War I in 1914 was precipitated by the rise of nationalism in Southeastern Europe as the Great Powers took up sides. The Allies defeated the Central Powers in 1918. During the Paris Peace Conference the Big Four imposed their terms in a series 

- For this question, the retriever found context passages related to World War I and World War II, but the reader model failed to extract a valid answer. Instead, it returned special transformer tokens such as [CLS] and [SEP].
- This demonstrates a limitation of extractive question answering systems. Even if the retrieved context is somewhat relevant, the reader may still fail when the answer is ambiguous, missing, or not clearly stated in the text.

In [60]:
question = "What is the tallest mountain in the world?"
context = get_context(question, top_k=3)
extract_answer(question, context)

[{'answer': 'mount everest',
  'context': "Tibet has some of the world's tallest mountains, with several of "
             'them making the top ten list. Mount Everest, located on the '
             'border with Nepal, is, at 8,848 metres (29,029 ft), the highest '
             'mountain on earth. Several major rivers have their source in the '
             'Tibetan Plateau (mostly in present-day Qinghai Province). These '
             'include the Yangtze, Yellow River, Indus River, Mekong, Ganges, '
             'Salween and the Yarlung Tsangpo River (Brahmaputra River). The '
             'Yarlung Tsangpo Grand Canyon, along the Yarlung Tsangpo River, '
             'is among the deepest and longest canyons in the world.',
  'score': 16.894065856933594},
 {'answer': '',
  'context': "The Union Internationale des Associations d'Alpinisme (UIAA) has "
             'defined a list of 82 "official" Alpine summits that reach at '
             'least 4,000 m (13,123 ft). The list includes

[{'answer': 'mount everest',
  'score': 16.894065856933594,
  'context': "Tibet has some of the world's tallest mountains, with several of them making the top ten list. Mount Everest, located on the border with Nepal, is, at 8,848 metres (29,029 ft), the highest mountain on earth. Several major rivers have their source in the Tibetan Plateau (mostly in present-day Qinghai Province). These include the Yangtze, Yellow River, Indus River, Mekong, Ganges, Salween and the Yarlung Tsangpo River (Brahmaputra River). The Yarlung Tsangpo Grand Canyon, along the Yarlung Tsangpo River, is among the deepest and longest canyons in the world."},
 {'answer': '',
  'score': 13.231218338012695,
  'context': 'The Union Internationale des Associations d\'Alpinisme (UIAA) has defined a list of 82 "official" Alpine summits that reach at least 4,000 m (13,123 ft). The list includes not only mountains, but also subpeaks with little prominence that are considered important mountaineering objectives. Below are

- For this question, the system performed much better. The retriever successfully found a relevant context passage containing the correct answer, and the reader model correctly extracted “Mount Everest” as the answer.
- The additional retrieved passages were related to mountains but did not contain the exact answer, which resulted in empty outputs or special tokens like [CLS]. This shows the system can return multiple related contexts, but only the most relevant passage usually produces the correct answer.
- Overall, this example demonstrates that the extractive question answering pipeline works well when the retriever finds a highly relevant context passage.

In [63]:
question = "Who invented the first web browser?"
context = get_context(question, top_k=3)
extract_answer(question, context)

[{'answer': 'sir tim berners - lee',
  'context': 'The first web browser was invented in 1990 by Sir Tim '
             'Berners-Lee. Berners-Lee is the director of the World Wide Web '
             "Consortium (W3C), which oversees the Web's continued "
             'development, and is also the founder of the World Wide Web '
             'Foundation. His browser was called WorldWideWeb and later '
             'renamed Nexus.',
  'score': 20.505218505859375},
 {'answer': 'marc andreessen',
  'context': 'In 1993, browser software was further innovated by Marc '
             'Andreessen with the release of Mosaic, "the world\'s first '
             'popular browser", which made the World Wide Web system easy to '
             "use and more accessible to the average person. Andreesen's "
             'browser sparked the internet boom of the 1990s. The introduction '
             'of Mosaic in 1993 – one of the first graphical web browsers – '
             'led to an explosion in web u

[{'answer': 'sir tim berners - lee',
  'score': 20.505218505859375,
  'context': "The first web browser was invented in 1990 by Sir Tim Berners-Lee. Berners-Lee is the director of the World Wide Web Consortium (W3C), which oversees the Web's continued development, and is also the founder of the World Wide Web Foundation. His browser was called WorldWideWeb and later renamed Nexus."},
 {'answer': 'marc andreessen',
  'score': 17.778457641601562,
  'context': 'In 1993, browser software was further innovated by Marc Andreessen with the release of Mosaic, "the world\'s first popular browser", which made the World Wide Web system easy to use and more accessible to the average person. Andreesen\'s browser sparked the internet boom of the 1990s. The introduction of Mosaic in 1993 – one of the first graphical web browsers – led to an explosion in web use. Andreessen, the leader of the Mosaic team at National Center for Supercomputing Applications (NCSA), soon started his own company, named Net

- For this question, the system performed very well. The retriever successfully retrieved a highly relevant context passage, and the reader model correctly extracted “Sir Tim Berners-Lee” as the answer with a high confidence score.
- The retriever also returned additional related passages about web browser history, such as Marc Andreessen and Mosaic. While these passages were relevant to the topic, they did not directly answer the question. One result again returned the special token [SEP],showing that lower-ranked contexts may still produce invalid outputs.
- Overall, this example demonstrates that the extractive question answering system works effectively when the retriever finds precise and relevant context passages.